- https://github.com/swan-cern/sparkmonitor

In [1]:
!ipython profile locate default

/home/jovyan/work/.ipython/profile_default


In [3]:
!ls -la $(ipython profile locate default)

total 636
drwxr-xr-x. 1 jovyan users     74 Jul 27 19:24 .
drwxr-xr-x. 1 jovyan users     30 Jun 24 19:44 ..
drwxr-xr-x. 1 jovyan users     10 Jul 27 11:47 db
-rw-r--r--. 1 jovyan users 647168 Jul 27 19:24 history.sqlite
drwxr-xr-x. 1 jovyan users      0 Jun 24 19:44 log
drwx------. 1 jovyan users      0 Jun 24 19:44 pid
drwx------. 1 jovyan users      0 Jun 24 19:44 security
drwxr-xr-x. 1 jovyan users     84 Jul 17 12:47 startup


In [5]:
!ipython profile create echo "c.InteractiveShellApp.extensions.append('sparkmonitor.kernelextension')" >> "$(ipython profile locate default)/ipython_kernel_config.py"

[ProfileCreate] Generating default config file: PosixPath('/home/jovyan/work/.ipython/profile_echo/ipython_config.py')
[ProfileCreate] Generating default config file: PosixPath('/home/jovyan/work/.ipython/profile_echo/ipython_kernel_config.py')


In [6]:
!ls -la $(ipython profile locate default)

total 636
drwxr-xr-x. 1 jovyan users    122 Jul 27 19:25 .
drwxr-xr-x. 1 jovyan users     54 Jul 27 19:25 ..
drwxr-xr-x. 1 jovyan users     10 Jul 27 11:47 db
-rw-r--r--. 1 jovyan users 647168 Jul 27 19:25 history.sqlite
-rw-r--r--. 1 jovyan users      0 Jul 27 19:25 ipython_kernel_config.py
drwxr-xr-x. 1 jovyan users      0 Jun 24 19:44 log
drwx------. 1 jovyan users      0 Jun 24 19:44 pid
drwx------. 1 jovyan users      0 Jun 24 19:44 security
drwxr-xr-x. 1 jovyan users     84 Jul 17 12:47 startup


In [7]:
!cat $(ipython profile locate default)/ipython_kernel_config.py

In [ ]:
ipython profile create

In [7]:
# https://github.com/swan-cern/sparkmonitor/blob/master/README.md
import os
from pathlib import Path

import pyspark
import sparkmonitor


def iter_spark_jar_dirs() -> list[Path]:
    candidates = []

    spark_home = os.environ.get("SPARK_HOME")
    if spark_home:
        candidates.append(Path(spark_home) / "jars")

    candidates.append(Path(pyspark.__file__).resolve().parent / "jars")
    return [path for path in candidates if path.exists()]


def resolve_listener_jar(sparkmonitor_dir: Path) -> Path:
    for jars_dir in iter_spark_jar_dirs():
        for jar in jars_dir.glob("spark-core_*.jar"):
            # spark-core_2.13-3.5.8.jar => scala=2.13, spark_major=3
            scala_ver, spark_ver = jar.name.split("_")[1].split("-")[:2]
            spark_major = spark_ver.split(".")[0]
            if spark_major == "3" and scala_ver == "2.12":
                return sparkmonitor_dir / "listener_spark3_2.12.jar"
            if spark_major == "3" and scala_ver == "2.13":
                return sparkmonitor_dir / "listener_spark3_2.13.jar"
            if spark_major == "4" and scala_ver == "2.13":
                return sparkmonitor_dir / "listener_spark4_2.13.jar"

    raise RuntimeError(
        "Could not detect Spark/Scala version from SPARK_HOME or the pyspark installation"
    )


sparkmonitor_dir = Path(sparkmonitor.__file__).resolve().parent
listener_jar = resolve_listener_jar(sparkmonitor_dir)

In [5]:
print(sparkmonitor_dir)

/opt/conda/lib/python3.13/site-packages/sparkmonitor


In [6]:
print(listener_jar)

/opt/conda/lib/python3.13/site-packages/sparkmonitor/listener_spark4_2.13.jar
